# 13.04 AWS — Bedrock · Knowledge Bases · 인프라 카탈로그

`langchain-aws` 한 패키지로 AWS 의 LLM 스택 전체에 접근한다. Bedrock chat / embeddings, Kendra · Knowledge Bases retriever, S3 loader, DynamoDB checkpointer, Neptune graph 까지 하나의 IAM 자격증명 체계로 묶인다.

## 학습 목표

- `langchain-aws` 가 노출하는 chat · retriever · loader · checkpointer · graph 클래스 매핑을 본다
- Bedrock 의 두 chat 인터페이스(`ChatBedrock` legacy invoke vs. `ChatBedrockConverse`) 를 구분한다
- Knowledge Bases / Kendra / S3 / DynamoDB 가 어느 카테고리(`05_retrievers/` · `04_document_loaders/` · `06_checkpointers/`) 로 깊이 다뤄지는지 매핑한다

## 언제 이 공급자를 고르나

- 이미 AWS 에서 운영 중이고 **VPC · IAM · KMS** 안에서 LLM 트래픽을 처리해야 할 때
- Anthropic Claude 를 **Bedrock 경유**로 받고 싶을 때(데이터 보존·리전 제어 목적)
- AWS Knowledge Bases · Kendra 같은 **관리형 retriever** 를 그대로 쓰고 싶을 때
- LangGraph 영속 상태를 **DynamoDB** 에 두고 싶을 때

## 13.04.1 환경 설정

필요 자격증명: `AWS_ACCESS_KEY_ID` + `AWS_SECRET_ACCESS_KEY` + `AWS_REGION` (또는 IAM Role/`~/.aws/credentials`). Bedrock 모델은 콘솔에서 **모델별 액세스 활성화**가 별도 절차임에 주의.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
assert os.environ["AWS_ACCESS_KEY_ID"], "AWS_ACCESS_KEY_ID 가 .env 에 필요합니다"

from langchain_aws import ChatBedrockConverse

probe = ChatBedrockConverse(
    model="anthropic.claude-3-5-sonnet-20241022-v2:0",
    region=os.environ.get("AWS_REGION", "us-east-1"),
    max_tokens=64,
)
print(probe.invoke("ping").content[:80])

## 13.04.2 공급자 제품군 전체 지도

| 영역 | 심볼 | 비고 / 링크 |
|------|------|-------------|
| Chat (Converse API, 권장) | `ChatBedrockConverse` | Anthropic/Meta/Amazon Nova 등 통합 인터페이스 |
| Chat (legacy invoke) | `ChatBedrock` | 모델별 페이로드 다름. 신규 코드는 Converse 권장 |
| Embeddings | `BedrockEmbeddings(model_id="amazon.titan-embed-text-v2:0")` | Titan / Cohere on Bedrock |
| Prompt caching | `BedrockPromptCachingMiddleware` | → `11_provider_middleware/06_bedrock_prompt_caching.ipynb` |
| Retriever (Knowledge Bases) | `AmazonKnowledgeBasesRetriever` | 관리형 RAG. → `05_retrievers/05_vendor_managed.ipynb` |
| Retriever (Kendra) | `AmazonKendraRetriever` | 엔터프라이즈 검색 |
| Loader (S3 file) | `S3FileLoader` | → `04_document_loaders/03_cloud_storage.ipynb` |
| Loader (S3 dir) | `S3DirectoryLoader` | prefix 단위 로딩 |
| Checkpointer | `DynamoDBSaver` (`langgraph-checkpoint-aws`) | → `06_checkpointers/` |
| Graph | `NeptuneGraph` · `NeptuneAnalyticsGraph` | Cypher / openCypher |
| Vector store | `InMemoryVectorStore` 대신 OpenSearch · S3 Vectors · Valkey | `langchain-aws` 본체엔 vectorstore 없음 |

임베딩 저장소(vectorstore) 는 `langchain-aws` 본체에 없으므로 OpenSearch / S3 Vectors / Valkey 패키지를 별도 사용한다.

## 13.04.3 기본 사용 — chat

신규 코드는 `ChatBedrockConverse` 사용. 모델 ID 는 `anthropic.claude-3-5-sonnet-...:0` / `meta.llama3-1-70b-instruct-v1:0` / `amazon.nova-lite-v1:0` 등.

In [ ]:
from langchain_aws import ChatBedrockConverse

llm = ChatBedrockConverse(
    model="anthropic.claude-3-5-sonnet-20241022-v2:0",
    region="us-east-1",
    temperature=0,
    max_tokens=256,
)
msg = llm.invoke("한국어로 Bedrock 의 핵심 가치를 한 문장 요약")
print(msg.content)

## 13.04.4 공급자 특화 기능

### Bedrock Converse API

Converse API 는 모델별 페이로드 차이를 표준화한 단일 엔드포인트(`/converse`)다. 도구 호출, 멀티턴, 시스템 프롬프트 형식이 Anthropic/Meta/Amazon Nova/Mistral 모두 동일.

| 구 인터페이스 (`ChatBedrock`) | Converse (`ChatBedrockConverse`) |
|--------------------------------|----------------------------------|
| 모델별로 `body` JSON 형태 다름 | 통합 schema |
| tool calling 지원 모델 한정 | 표준 tool spec |
| 신규 모델 미지원 가능 | 신규 모델 우선 지원 |

### Knowledge Bases retriever-as-a-service

Bedrock Knowledge Bases 는 S3 → 청킹 → 임베딩(Titan) → OpenSearch 자동 인덱싱 + retrieve API 까지 관리형 제공. `AmazonKnowledgeBasesRetriever(knowledge_base_id="...")` 한 줄로 LangChain 체인에 결합.

```python
from langchain_aws import AmazonKnowledgeBasesRetriever
retriever = AmazonKnowledgeBasesRetriever(
    knowledge_base_id="AAAA1234",
    region_name="us-east-1",
    retrieval_config={"vectorSearchConfiguration": {"numberOfResults": 5}},
)
docs = retriever.invoke("매출 정책")
```

### prompt caching

Bedrock 위 Claude/Nova 의 prompt cache 는 `BedrockPromptCachingMiddleware` 로 따로 처리. Anthropic 직접 API 와 파라미터 의미는 거의 같지만 **Nova 는 5m TTL 만 지원**(1h 미지원). → `11_provider_middleware/06_bedrock_prompt_caching.ipynb` 참고.

## 13.04.5 가격·한도·리전

| 항목 | 값 (2025 공개치) |
|------|------------------|
| 리전 | Bedrock: us-east-1 / us-west-2 / ap-northeast-1(도쿄) / ap-northeast-2(서울, 일부 모델) / eu-central-1 등 — 모델별 가용 리전 다름 |
| 모델 액세스 | 콘솔 → Bedrock → Model access 에서 **모델별 활성화 신청** 필요(Anthropic Claude 등) |
| 가격 | 모델별 1k 입력/출력 토큰. Claude Sonnet 3.5 기준 입력 $0.003/1k, 출력 $0.015/1k (us-east-1) |
| Knowledge Bases | 임베딩+retrieve 호출 별도 과금. OpenSearch Serverless OCU 비용 추가 |
| Kendra | 엔터프라이즈 검색 — 인덱스 단위 시간당 과금($1.125+/h) |
| Bedrock prompt cache | cache write $0.30/1M cache token, read $0.03/1M (Claude 기준, write 가 비싸지만 read 가 90% 할인) |
| 데이터 보존 | Bedrock 은 AWS 계정 안에서 끝남(모델 학습 미사용). VPC endpoint 로 인터넷 우회 가능 |

최신 단가: https://aws.amazon.com/bedrock/pricing/ — 모델별·리전별로 자주 변동.

## 핵심 정리

- `langchain-aws` 한 패키지에 chat · embed · retriever · loader · graph 가 모이고, IAM 자격 한 벌로 모두 인증
- Bedrock chat 은 신규 코드라면 무조건 **Converse API** (`ChatBedrockConverse`) 권장
- Knowledge Bases 는 임베딩·인덱싱·retrieve 까지 관리형 — 자체 vectorstore 운영 부담을 없앤다
- vectorstore 는 본체에 없음 — OpenSearch / S3 Vectors / Valkey 별도 패키지 사용

## 다음

- chat 깊은 구현: `07_integration/01_chat_models/05_bedrock.ipynb`
- prompt caching: `07_integration/11_provider_middleware/06_bedrock_prompt_caching.ipynb`
- Knowledge Bases retriever: `07_integration/05_retrievers/05_vendor_managed.ipynb`
- 다음 공급자 카탈로그: `05_microsoft.ipynb`

## 참고

- 공식 통합 페이지: https://docs.langchain.com/oss/python/integrations/providers/aws
- Bedrock 사용자 가이드: https://docs.aws.amazon.com/bedrock/
- Knowledge Bases 콘솔: https://console.aws.amazon.com/bedrock/home#/knowledge-bases